In [0]:
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
def upsert_delta_table(df_new, table_name, join_condition):
    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        delta_table.alias("old").merge(
            df_new.alias("new"),
            join_condition
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
    else:
        df_new.write.mode("overwrite").saveAsTable(table_name)

In [0]:
df_bronze = spark.table("bronze_clinvar.bronze_clinvar_raw")

In [0]:
df_silver_variants = df_bronze.select(
    F.col("_VariationID").alias("variation_id"),
    F.col("_VariationName").alias("variation_name"),
    F.col("ClassifiedRecord.SimpleAllele.Name").alias("hgvs_name"),
    F.upper(F.col("_VariationType")).alias("variation_type"),
    F.col("_Accession").alias("vcv_accession"),
    F.col("_Version").cast("int").alias("vcv_version"),
    F.col("_DateLastUpdated").alias("date_last_updated"),
    F.col("RecordStatus").alias("status"),
    F.col("_metadata.file_path").alias("source_file"),
    "ingestion_date"
)

upsert_delta_table(df_silver_variants, "silver_clinvar.silver_clinvar_variants", "old.variation_id = new.variation_id")


In [0]:

silver_locations_details = df_bronze.select(
    F.col("_VariationID").alias("variation_id"),
    F.explode("ClassifiedRecord.SimpleAllele.Location.SequenceLocation").alias("loc"),
    "ingestion_date"
).select(
    "variation_id",
    F.col("loc._Assembly").alias("assembly"),
    F.col("loc._Chr").alias("chromosome"),
    F.col("loc._Start").cast("long").alias("start_pos"),
    F.col("loc._Stop").cast("long").alias("stop_pos"),
    "ingestion_date"
).filter(F.col("assembly") == "GRCh38")

upsert_delta_table(silver_locations_details, "silver_clinvar.silver_clinvar_locations", "old.variation_id = new.variation_id")

In [0]:
df_genes = df_bronze.select(
    F.col("_VariationID").alias("variation_id"),
    F.explode("ClassifiedRecord.SimpleAllele.GeneList.Gene").alias("gene"),
    "ingestion_date"
).select(
    "variation_id",
    F.col("gene._GeneID").alias("gene_id"),
    F.col("gene._Symbol").alias("gene_symbol"),
    F.upper(F.col("gene._FullName")).alias("gene_name"),
    F.col("gene.Location.SequenceLocation._start").alias("gene_start"),
    F.col("gene.Location.SequenceLocation._stop").alias("gene_stop")
)

df_genes = df_genes.dropDuplicates(["variation_id", "gene_id"])
upsert_delta_table(df_genes, "silver_clinvar.silver_clinvar_genes", "old.variation_id = new.variation_id AND old.gene_id = new.gene_id")


In [0]:
df_traits = df_bronze.select(
    F.col("_VariationID").alias("variation_id"),
    F.explode("ClassifiedRecord.TraitMappingList.TraitMapping").alias("trait"),
    "ingestion_date"
).select(
    "variation_id",
    F.col("trait._ClinicalAssertionID").alias("clinical_assertion_id"),
    F.upper(F.col("trait._TraitType")).alias("trait_type"),
    F.col("trait._MappingValue").alias("mapping_value"),
    F.col("trait._MappingRef").alias("mapping_ref"),
    F.col("trait.MedGen._CUI").alias("medgen_cui"),
    F.upper(F.col("trait.MedGen._Name")).alias("medgen_name"),
    "ingestion_date"
).withColumn(
    "mapping_value", F.coalesce(F.col("mapping_value"), F.lit("NOT PROVIDED"))
).withColumn(
    "medgen_name", F.coalesce(F.col("medgen_name"), F.lit("NOT PROVIDED"))
)

df_traits = df_traits.dropDuplicates(["variation_id", "clinical_assertion_id"])
upsert_delta_table(df_traits, "silver_clinvar.silver_clinvar_traits", "old.variation_id = new.variation_id AND old.clinical_assertion_id = new.clinical_assertion_id")

In [0]:

df_clinical_summary = df_bronze.select(
    F.col("_VariationID").alias("variation_id"),
    F.upper(F.col("ClassifiedRecord.Classifications.GermlineClassification.Description")).alias("aggregate_clinical_significance"),
    F.upper(F.col("ClassifiedRecord.Classifications.GermlineClassification.ReviewStatus")).alias("aggregate_review_status"),
    F.col("ClassifiedRecord.Classifications.GermlineClassification._DateLastEvaluated").cast("date").alias("last_evaluated_global"),
    "ingestion_date"
).filter(
    F.trim(F.col("aggregate_review_status")).isin(
        'REVIEWED BY EXPERT PANEL', 
        'CRITERIA PROVIDED, MULTIPLE SUBMITTERS, NO CONFLICTS', 
        'PRACTICE GUIDELINE', 
        'CRITERIA PROVIDED, SINGLE SUBMITTER'
    )
)

# Upsert na tabela de sumário clínico
upsert_delta_table(df_clinical_summary, "silver_clinvar.silver_clinvar_clinical_summary", "old.variation_id = new.variation_id")

In [0]:
df_assertions = df_bronze.select(
    F.col("_VariationID").alias("variation_id"),
    F.explode("ClassifiedRecord.ClinicalAssertionList.ClinicalAssertion").alias("assertion"),
    "ingestion_date"
).select(
    "variation_id",
    F.col("assertion._ID").alias("assertion_id"),
    F.col("assertion._DateCreated").cast("date").alias("date_created"),
    F.col("assertion._DateLastUpdated").cast("date").alias("date_last_updated"),
    F.col("assertion._ContributesToAggregateClassification").cast("boolean").alias("contributes_to_classification"),
    F.col("assertion.ClinVarAccession._Accession").alias("clinvar_accession"),
    F.col("assertion.ClinVarAccession._Type").alias("type"),
    F.upper(F.col("assertion.ClinVarAccession._SubmitterName")).alias("submitter_name"),
    F.upper(F.col("assertion.ClinVarAccession._OrganizationCategory")).alias("organization_category"),
    F.col("assertion.Classification._DateLastEvaluated").alias("classification_last_evaluated"),
    F.upper(F.col("assertion.Classification.ReviewStatus._VALUE")).alias("classification_review_status"),
    F.upper(F.col("assertion.Classification.GermlineClassification._VALUE")).alias("classification_germline_classification"),
    F.upper(F.col("assertion.ObservedInList.ObservedIn.Method.MethodType").getItem(0).getItem(0)).alias("method_type"),
    F.upper(F.col("assertion.ObservedInList.ObservedIn.Method.TypePlatform").getItem(0).getItem(0)).alias("method_type_platform"),
    F.col("assertion.Classification.Citation.ID._VALUE").alias("citation_id"),
    F.col("assertion.Classification.Citation.ID._Source").alias("citation_source"),
    "ingestion_date"
).filter(
        F.col("classification_review_status").isin(
            'REVIEWED BY EXPERT PANEL',
            'PRACTICE GUIDELINE',
            'CRITERIA PROVIDED, MULTIPLE SUBMITTERS, NO CONFLICTS',
            'CRITERIA PROVIDED, SINGLE SUBMITTER'
        )
    )

upsert_delta_table(df_assertions, "silver_clinvar.silver_clinvar_assertions", "old.assertion_id = new.assertion_id")
